# Imports

In [86]:
import os
import sys
from pathlib import Path
from IPython.display import display
import pandas as pd
PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current working directory:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("data_and_annotation exists:", (PROJECT_ROOT / "src" / "data_and_annotation.py").exists())
print("__init__.py exists:", (PROJECT_ROOT / "src" / "__init__.py").exists())



Current working directory: c:\DL_Env\Projects\NLP Assignment 2\Sequence Labeling POS Tagging & NER
src exists: True
data_and_annotation exists: True
__init__.py exists: True


In [87]:
from src.data_and_annotation import (
    prepare_part2_data,
    label_distribution,
    topic_distribution
)

print("Imported functions successfully.")

Imported functions successfully.


In [88]:
BASE_DIR = "."
CLEANED_PATH = os.path.join(BASE_DIR, "cleaned.txt")
META_PATH = os.path.join(BASE_DIR, "metadata.json")
DATA_DIR = os.path.join(BASE_DIR, "data")

print("cleaned exists:", os.path.exists(CLEANED_PATH))
print("metadata exists:", os.path.exists(META_PATH))
print("data dir exists before creation:", os.path.exists(DATA_DIR))

cleaned exists: True
metadata exists: True
data dir exists before creation: True


## Part 2 data preparation

In [89]:
bundle = prepare_part2_data(
    cleaned_path=CLEANED_PATH,
    metadata_path=META_PATH,
    data_dir=DATA_DIR,
    target_size=500
)

articles = bundle["articles"]
selected_df = bundle["selected_df"]
annotated_examples = bundle["annotated_examples"]
train_data = bundle["train_data"]
val_data = bundle["val_data"]
test_data = bundle["test_data"]

pos2idx = bundle["pos2idx"]
idx2pos = bundle["idx2pos"]
ner2idx = bundle["ner2idx"]
idx2ner = bundle["idx2ner"]

print("Part 2A data preparation completed.")
print("Selected sentences:", len(annotated_examples))
print("Train / Val / Test:", len(train_data), len(val_data), len(test_data))

Part 2A data preparation completed.
Selected sentences: 500
Train / Val / Test: 350 75 75


## Topic distribution

In [90]:
print("Selected sentence topic distribution:")
display(selected_df["topic"].value_counts().to_frame("count"))

print("\nTrain topic distribution:")
display(topic_distribution(train_data).to_frame("count"))

print("\nValidation topic distribution:")
display(topic_distribution(val_data).to_frame("count"))

print("\nTest topic distribution:")
display(topic_distribution(test_data).to_frame("count"))

Selected sentence topic distribution:


,count
topic,
politics,202
health_society,107
economy,92
international,66
sports,33



Train topic distribution:


,count
politics,142
health_society,75
economy,64
international,46
sports,23



Validation topic distribution:


,count
politics,30
health_society,16
economy,14
international,10
sports,5



Test topic distribution:


,count
politics,30
health_society,16
economy,14
international,10
sports,5


## POS and NER label distributions

In [91]:
print("POS label distribution (all selected data):")
display(label_distribution(annotated_examples, task="pos").to_frame("count"))

print("\nNER label distribution (all selected data):")
display(label_distribution(annotated_examples, task="ner").to_frame("count"))

POS label distribution (all selected data):


,count
NOUN,6759
POST,1949
ADJ,1013
VERB,876
PRON,874
CONJ,505
DET,149
NUM,144
ADV,91
UNK,31



NER label distribution (all selected data):


,count
O,11670
B-LOC,290
B-ORG,196
B-PER,116
I-ORG,67
I-PER,36
I-LOC,15
B-MISC,8


## Preview annotations

In [92]:


sample = annotated_examples[0]
preview_rows = []

for tok, pos, ner in zip(sample["tokens"], sample["pos_tags"], sample["ner_tags"]):
    preview_rows.append({
        "token": tok,
        "pos": pos,
        "ner": ner
    })

print("Sample annotation preview:")
display(pd.DataFrame(preview_rows))

Sample annotation preview:


,token,pos,ner
0,وہ,PRON,O
1,جماعت,NOUN,O
2,کے,POST,O
3,قائد,NOUN,O
4,رہے,VERB,O
5,اور,CONJ,O
6,اب,ADV,O
7,صدر,NOUN,O
8,ہیں,VERB,O
9,اور,CONJ,O


## saved CoNLL files

In [93]:
required_files = [
    os.path.join(DATA_DIR, "pos_train.conll"),
    os.path.join(DATA_DIR, "pos_test.conll"),
    os.path.join(DATA_DIR, "ner_train.conll"),
    os.path.join(DATA_DIR, "ner_test.conll"),
]

for f in required_files:
    print(f, "FOUND" if os.path.exists(f) else "MISSING")

.\data\pos_train.conll FOUND
.\data\pos_test.conll FOUND
.\data\ner_train.conll FOUND
.\data\ner_test.conll FOUND


# Import dataset utilities

In [94]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from datasets import (
    load_word2idx,
    load_embeddings,
    build_model_vocab,
    SequenceTaggingDataset,
    make_collate_fn,
    PAD_TOKEN,
    UNK_TOKEN,
    PAD_LABEL
)

WORD2IDX_PATH = os.path.join(BASE_DIR, "word2idx.json")
EMB_PATH = os.path.join(BASE_DIR, "embeddings_w2v.npy")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dataset utilities imported successfully.")
print("Using device:", DEVICE)

Dataset utilities imported successfully.
Using device: cuda


## Loading Part 1 vocabulary and embeddings

In [95]:
base_word2idx = load_word2idx(WORD2IDX_PATH)
base_embeddings = load_embeddings(EMB_PATH)

word2idx, idx2word, embedding_matrix = build_model_vocab(base_word2idx, base_embeddings)

print("Original embedding matrix shape :", base_embeddings.shape)
print("Expanded embedding matrix shape :", embedding_matrix.shape)
print("Vocabulary size with PAD        :", len(word2idx))
print("PAD token id                    :", word2idx[PAD_TOKEN])
print("UNK token id                    :", word2idx[UNK_TOKEN])

Original embedding matrix shape : (4387, 100)
Expanded embedding matrix shape : (4388, 100)
Vocabulary size with PAD        : 4388
PAD token id                    : 0
UNK token id                    : 1


## Build POS and NER datasets

In [96]:
pos_train_dataset = SequenceTaggingDataset(train_data, word2idx, pos2idx, task="pos")
pos_val_dataset = SequenceTaggingDataset(val_data, word2idx, pos2idx, task="pos")
pos_test_dataset = SequenceTaggingDataset(test_data, word2idx, pos2idx, task="pos")

ner_train_dataset = SequenceTaggingDataset(train_data, word2idx, ner2idx, task="ner")
ner_val_dataset = SequenceTaggingDataset(val_data, word2idx, ner2idx, task="ner")
ner_test_dataset = SequenceTaggingDataset(test_data, word2idx, ner2idx, task="ner")

print("POS train/val/test sizes:", len(pos_train_dataset), len(pos_val_dataset), len(pos_test_dataset))
print("NER train/val/test sizes:", len(ner_train_dataset), len(ner_val_dataset), len(ner_test_dataset))

POS train/val/test sizes: 350 75 75
NER train/val/test sizes: 350 75 75


## Build dataloaders

In [97]:
BATCH_SIZE = 32
collate_fn = make_collate_fn(pad_token_id=word2idx[PAD_TOKEN])

pos_train_loader = DataLoader(pos_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
pos_val_loader = DataLoader(pos_val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
pos_test_loader = DataLoader(pos_test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

ner_train_loader = DataLoader(ner_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
ner_val_loader = DataLoader(ner_val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
ner_test_loader = DataLoader(ner_test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("Dataloaders created successfully.")

Dataloaders created successfully.


## Inspect one POS batch

In [98]:
batch = next(iter(pos_train_loader))

print("POS batch shapes")
print("input_ids      :", batch["input_ids"].shape)
print("label_ids      :", batch["label_ids"].shape)
print("attention_mask :", batch["attention_mask"].shape)
print("lengths        :", batch["lengths"].shape)

print("\nFirst sequence tokens:")
print(batch["tokens"][0])

print("\nFirst sequence POS labels:")
print(batch["labels"][0])

POS batch shapes
input_ids      : torch.Size([32, 40])
label_ids      : torch.Size([32, 40])
attention_mask : torch.Size([32, 40])
lengths        : torch.Size([32])

First sequence tokens:
['اس', 'سلسلے', 'میں', 'ادارے', 'نے', 'قواعد', 'و', 'ضوابط', 'بھی', 'جاری', 'کیے', 'ہیں،', 'جن', 'کے', 'تحت', 'سولر', 'سے', 'بجلی', 'پیدا', 'کر', 'کے', 'حکومت', 'کو', 'فروخت', 'کرنے', 'والے', 'صارفین', 'کے', 'لیے', 'نئے', 'نرخ', 'مقرر', 'کیے', 'گئے', 'ہیں۔']

First sequence POS labels:
['PRON', 'NOUN', 'PRON', 'NOUN', 'NOUN', 'NOUN', 'UNK', 'NOUN', 'ADJ', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'POST', 'POST', 'NOUN', 'POST', 'ADJ', 'NOUN', 'VERB', 'POST', 'NOUN', 'POST', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'POST', 'POST', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'NOUN']


## Inspect one NER batch

In [100]:
batch = next(iter(ner_train_loader))

print("NER batch shapes")
print("input_ids      :", batch["input_ids"].shape)
print("label_ids      :", batch["label_ids"].shape)
print("attention_mask :", batch["attention_mask"].shape)
print("lengths        :", batch["lengths"].shape)

print("\nFirst sequence tokens:")
print(batch["tokens"][0])

print("\nFirst sequence NER labels:")
print(batch["labels"][0])

NER batch shapes
input_ids      : torch.Size([32, 40])
label_ids      : torch.Size([32, 40])
attention_mask : torch.Size([32, 40])
lengths        : torch.Size([32])

First sequence tokens:
['حافظ', 'گل', 'بہادر', 'کالعدم', 'ٹی', 'ٹی', 'پی', 'کے', 'اپنے', 'گروہ', 'کے', 'سربراہ', 'ہیں', 'جبکہ', 'سربکف', 'مہمند', 'جماعت', 'الاحرار', 'کے', 'سربراہ', 'ہونے', 'کے', 'ساتھ', 'ساتھ', 'کالعدم', 'ٹی', 'ٹی', 'پی', 'کا', 'بھی', 'حصہ', 'ہیں۔']

First sequence NER labels:
['O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O']


## Save processed embedding matrix for reuse

In [101]:
np.save(os.path.join(BASE_DIR, "embedding_matrix_part2.npy"), embedding_matrix.astype(np.float32))
print("Saved embedding_matrix_part2.npy successfully.")

Saved embedding_matrix_part2.npy successfully.
